In [1]:
# [CELL 1] IMPORTS AND STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil
import statistics

LAST_NAME = "Malitao"
STUDENT_ID = "TUPM-26-1135"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())

TARGET_YEAR = 2015 + SEED_DIGIT

BASE_DIR = Path.cwd()
DATASET_FILE = BASE_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
OUTPUT_DIR = BASE_DIR

print("STUDENT CONFIGURATION")
print("---------------------")
print("Last Name:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Target Year:", TARGET_YEAR)
print("Dataset Found:", DATASET_FILE.exists())
print("Dataset Path:", DATASET_FILE)

STUDENT CONFIGURATION
---------------------
Last Name: Malitao
Student ID: TUPM-26-1135
Seed Digit: 5
Target Year: 2020
Dataset Found: True
Dataset Path: /Applications/MALITAO/Lab6_File_Handling_Malitao/Lab6_File_Handling_Additional_Exercises/Exercise3/Global_Cybersecurity_Threats_2015-2024.csv


In [2]:
# [CELL 2] DATASET INSPECTION AND VALIDATION

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    actual_columns = reader.fieldnames
    total_dataset_records = sum(1 for row in reader)

required_columns = [
    "Year",
    "Attack Type",
    "Financial Loss (in Million $)"
]

missing_columns = [
    column for column in required_columns
    if column not in actual_columns
]

print("DATASET INSPECTION")
print("------------------")
print("Dataset File:", DATASET_FILE.name)
print("Total Records:", total_dataset_records)
print("Available Columns:", actual_columns)

print("\nCOLUMN VALIDATION")
print("-----------------")
print("Required Columns:", required_columns)
print("Missing Columns:", missing_columns)
print("Dataset Valid:", len(missing_columns) == 0)

DATASET INSPECTION
------------------
Dataset File: Global_Cybersecurity_Threats_2015-2024.csv
Total Records: 3000
Available Columns: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']

COLUMN VALIDATION
-----------------
Required Columns: ['Year', 'Attack Type', 'Financial Loss (in Million $)']
Missing Columns: []
Dataset Valid: True


In [3]:
# [CELL 3] FILTER INCIDENTS AND VALIDATE FINANCIAL LOSSES

selected_incidents = []
invalid_records = 0
total_records_read = 0

with open(DATASET_FILE, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records_read += 1

        try:
            year = int(row["Year"].strip())
        except (ValueError, AttributeError):
            invalid_records += 1
            continue

        if year != TARGET_YEAR:
            continue

        try:
            financial_loss = float(
                row["Financial Loss (in Million $)"].strip()
            )

            if financial_loss < 0:
                raise ValueError

        except (ValueError, AttributeError):
            invalid_records += 1
            continue

        incident = dict(row)
        incident["Financial Loss (in Million $)"] = financial_loss

        selected_incidents.append(incident)

print("INCIDENT FILTERING RESULTS")
print("--------------------------")
print("Total Records Read:", total_records_read)
print("Target Year:", TARGET_YEAR)
print("Selected Incidents:", len(selected_incidents))
print("Invalid Records:", invalid_records)

INCIDENT FILTERING RESULTS
--------------------------
Total Records Read: 3000
Target Year: 2020
Selected Incidents: 315
Invalid Records: 0


In [4]:
# [CELL 4] CYBERSECURITY INCIDENT ANALYSIS

attack_type_counts = {}

for incident in selected_incidents:
    attack_type = incident["Attack Type"]

    attack_type_counts[attack_type] = (
        attack_type_counts.get(attack_type, 0) + 1
    )

financial_losses = [
    incident["Financial Loss (in Million $)"]
    for incident in selected_incidents
]

if financial_losses:
    total_financial_loss = sum(financial_losses)
    average_financial_loss = statistics.mean(financial_losses)

    # Stable sorting preserves original order when losses are tied.
    top_five_incidents = sorted(
        selected_incidents,
        key=lambda incident: incident["Financial Loss (in Million $)"],
        reverse=True
    )[:5]

else:
    total_financial_loss = 0
    average_financial_loss = 0
    top_five_incidents = []

print("ATTACK TYPE SUMMARY")
print("-------------------")

for attack_type, count in sorted(attack_type_counts.items()):
    print(f"{attack_type}: {count}")

print("\nFINANCIAL LOSS SUMMARY")
print("----------------------")
print(f"Total Financial Loss: ${total_financial_loss:.2f} million")
print(f"Average Financial Loss: ${average_financial_loss:.2f} million")
print("Total Incidents:", len(selected_incidents))

print("\nTOP FIVE HIGHEST-LOSS INCIDENTS")
print("--------------------------------")

for rank, incident in enumerate(top_five_incidents, start=1):
    print(
        f"{rank}. {incident['Attack Type']} | "
        f"{incident['Country']} | "
        f"${incident['Financial Loss (in Million $)']:.2f} million"
    )

ATTACK TYPE SUMMARY
-------------------
DDoS: 56
Malware: 37
Man-in-the-Middle: 47
Phishing: 61
Ransomware: 59
SQL Injection: 55

FINANCIAL LOSS SUMMARY
----------------------
Total Financial Loss: $15767.95 million
Average Financial Loss: $50.06 million
Total Incidents: 315

TOP FIVE HIGHEST-LOSS INCIDENTS
--------------------------------
1. Ransomware | Brazil | $99.90 million
2. DDoS | Japan | $99.83 million
3. DDoS | USA | $99.81 million
4. SQL Injection | India | $99.72 million
5. Phishing | USA | $99.64 million


In [5]:
# [CELL 5] EXPORT RESULTS AND GENERATE TEXT REPORT

FILTERED_CSV = OUTPUT_DIR / "filtered_cybersecurity_incidents.csv"
TOP_FIVE_JSON = OUTPUT_DIR / "top_five_incidents.json"
REPORT_TXT = OUTPUT_DIR / "cybersecurity_report.txt"

# Export all selected incidents to CSV
if selected_incidents:
    fieldnames = list(selected_incidents[0].keys())

    with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(selected_incidents)

else:
    with open(FILTERED_CSV, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(required_columns)

# Export top five incidents to JSON
with open(TOP_FIVE_JSON, "w", encoding="utf-8") as file:
    json.dump(top_five_incidents, file, indent=4)

# Generate text report
report_lines = [
    "CYBERSECURITY INCIDENT REPORT",
    "============================",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Target Year: {TARGET_YEAR}",
    "",
    "INCIDENT SUMMARY",
    "-----------------",
    f"Total Incidents: {len(selected_incidents)}",
    f"Invalid Records: {invalid_records}",
    "",
    "ATTACK TYPE COUNTS",
    "------------------"
]

for attack_type, count in sorted(attack_type_counts.items()):
    report_lines.append(f"{attack_type}: {count}")

report_lines.extend([
    "",
    "FINANCIAL LOSS SUMMARY",
    "----------------------",
    f"Total Financial Loss: ${total_financial_loss:.2f} million",
    f"Average Financial Loss: ${average_financial_loss:.2f} million",
    "",
    "TOP FIVE HIGHEST-LOSS INCIDENTS",
    "--------------------------------"
])

for rank, incident in enumerate(top_five_incidents, start=1):
    report_lines.append(
        f"{rank}. {incident['Attack Type']} - "
        f"{incident['Country']} - "
        f"${incident['Financial Loss (in Million $)']:.2f} million"
    )

with open(REPORT_TXT, "w", encoding="utf-8") as file:
    file.write("\n".join(report_lines))

print("FILE EXPORT RESULTS")
print("-------------------")
print("Filtered CSV:", FILTERED_CSV.name)
print("Top Five JSON:", TOP_FIVE_JSON.name)
print("Text Report:", REPORT_TXT.name)
print("Selected Incidents Exported:", len(selected_incidents))
print("Top Five Incidents Exported:", len(top_five_incidents))
print("Text Report Created: SUCCESS")

FILE EXPORT RESULTS
-------------------
Filtered CSV: filtered_cybersecurity_incidents.csv
Top Five JSON: top_five_incidents.json
Text Report: cybersecurity_report.txt
Selected Incidents Exported: 315
Top Five Incidents Exported: 5
Text Report Created: SUCCESS


In [6]:
# [CELL 6] BACKUP AND FINAL FILE VERIFICATION

BACKUP_JSON = OUTPUT_DIR / "top_five_incidents_backup.json"

# Create JSON backup
shutil.copy2(TOP_FIVE_JSON, BACKUP_JSON)

# Verify CSV record count
with open(FILTERED_CSV, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    exported_count = sum(1 for row in reader)

# Verify JSON record count
with open(TOP_FIVE_JSON, "r", encoding="utf-8") as file:
    exported_top_five = json.load(file)

# Verify backup contents
with open(BACKUP_JSON, "r", encoding="utf-8") as file:
    backup_top_five = json.load(file)

print("FILE VERIFICATION")
print("-----------------")

for file_path in [
    FILTERED_CSV,
    TOP_FIVE_JSON,
    REPORT_TXT,
    BACKUP_JSON
]:
    print(
        file_path.name,
        "- VERIFIED" if file_path.exists() else "- NOT FOUND"
    )

print("\nRECORD COUNT VERIFICATION")
print("-------------------------")
print("Selected Incidents:", len(selected_incidents))
print("Exported Incidents:", exported_count)
print("Count Matches:", exported_count == len(selected_incidents))
print("Top Five Selected:", len(top_five_incidents))
print("Top Five Exported:", len(exported_top_five))
print("Top Five Count Matches:", len(exported_top_five) == 5)

print("\nBACKUP VERIFICATION")
print("-------------------")
print("Backup Exists:", BACKUP_JSON.exists())
print("Backup Matches:", backup_top_five == exported_top_five)
print("Backup File Size:", BACKUP_JSON.stat().st_size, "bytes")

FILE VERIFICATION
-----------------
filtered_cybersecurity_incidents.csv - VERIFIED
top_five_incidents.json - VERIFIED
cybersecurity_report.txt - VERIFIED
top_five_incidents_backup.json - VERIFIED

RECORD COUNT VERIFICATION
-------------------------
Selected Incidents: 315
Exported Incidents: 315
Count Matches: True
Top Five Selected: 5
Top Five Exported: 5
Top Five Count Matches: True

BACKUP VERIFICATION
-------------------
Backup Exists: True
Backup Matches: True
Backup File Size: 2152 bytes
